<a href="https://colab.research.google.com/github/KadariPavani/ai-ml-kickoff/blob/main/AI_ML_Kickoff.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI and ML Kickoff: Easy Hands-on Lab
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/ai-ml-kickoff/blob/main/AI_ML_Kickoff_Lab.ipynb)

**How this lab works:** every stage has
1. **Easy notes** in plain English, with an everyday comparison
2. **Try it**: short code you just run (change the boxes on the right of a cell to try your own input)
3. **Your turn**: a small task, with a hidden answer
4. **Quick check**: 2 questions to test yourself

| Stage | Topic | You will |
|---|---|---|
| 1 | AI in the Real World | check if a sentence is happy or sad, catch spam |
| 2 | The AI Map | predict pass/fail, catch a server problem |
| 3 | Free Models | chat with a free AI model |
| 4 | How LLMs Work | see tokens and the model's next-word guesses |
| 5 | Benchmarks | score models and pick one |
| 6 | GitHub Skills | write a skill and see the difference |
| 7 | RAG + OKF | ask questions from your own notes |
| 8 | Night project | build your Knowledge RAG Agent |

# Stage 0: Setup (2 minutes)
1. **File > Save a copy in Drive** (so you can edit your own copy).
2. Click the play button on the cell below, or press **Shift + Enter**.
3. Cells with a hidden body (like *Show answer*) are hidden on purpose. Think first, then run them.

> The first time a model is used it downloads (about 1 GB in total for today). After that it is fast.

In [ ]:
#@title Setup: just run me { display-mode: "form" }
import time, re, os, glob
import numpy as np, pandas as pd, matplotlib.pyplot as plt

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # a small, free AI model (Apache-2.0 license)
_chat = None

def ask(question, rules=None, words=150):
    """Ask the free AI model a question. Returns the answer and the time taken."""
    global _chat
    if _chat is None:
        from transformers import pipeline
        print("Loading the free model (first time only)...")
        _chat = pipeline("text-generation", model=MODEL)
    messages = ([{"role": "system", "content": rules}] if rules else []) + [{"role": "user", "content": question}]
    start = time.time()
    reply = _chat(messages, max_new_tokens=words, do_sample=False)[0]["generated_text"]
    text = reply[-1]["content"] if isinstance(reply, list) else str(reply)
    return text.strip(), round(time.time() - start, 1)

print("Setup done. You are ready!")

# Stage 1: AI in the Real World
> **In one line:** AI is already inside the apps you use every day; it either **predicts** something or **generates** something new.

### Easy notes
**What is AI?** Teaching computers to do things that normally need human thinking: understanding, recognising, predicting, recommending, creating.

**Where is it?** Everywhere:

| App | What the AI does |
|---|---|
| YouTube | guesses the next video you will like |
| Google Maps | predicts when you will arrive |
| Gmail | moves spam out of your inbox |
| UPI / bank app | flags a strange payment |
| Phone keyboard | guesses your next word |
| Face unlock | recognises your face |

**Two big jobs of AI**
- **Prediction:** gives a label or a number. *Is this spam? Will it rain?*
- **Generation:** creates something new. *Write a poem. Draw a logo.*

**Remember:** AI can be confidently wrong. Always check its output before you use it, like checking a friend's notes before the exam.

## Try it 1: Is this sentence happy or sad?
This uses a real AI model that already learned from thousands of movie reviews. Type any sentence in the box on the right of the cell and run it.

In [ ]:
from transformers import pipeline
mood = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")

sentence = "The canteen food was amazing today!"  #@param {type:"string"}
result = mood(sentence)[0]
print(f"{sentence}\n-> {result['label']} (confidence {result['score']:.0%})")

**What you will see:** POSITIVE or NEGATIVE with a confidence. Try *My train got cancelled again* and *The class was okay*. The last one is tricky: this model only knows happy or sad.

## Try it 2: Build a spam catcher in 10 lines
We show the computer some messages that we already marked as spam or normal. It **learns** the pattern by itself.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

examples = {
    "You won a free iPhone, click now": "SPAM",      "Claim your cash prize today": "SPAM",
    "Your account is blocked, send OTP": "SPAM",     "Earn money from home, no work": "SPAM",
    "Free recharge offer, click the link": "SPAM",   "Lottery winner! Send bank details": "SPAM",
    "Class moved to room 204 tomorrow": "normal",    "Please send the assignment PDF": "normal",
    "Can we meet after the lab?": "normal",          "Exam timetable is attached": "normal",
    "Call mom when you reach": "normal",             "Project review is on Friday": "normal",
}
words = CountVectorizer().fit(examples.keys())
catcher = MultinomialNB().fit(words.transform(examples.keys()), list(examples.values()))

message = "Click here to claim your free prize"  #@param {type:"string"}
print(message, "->", catcher.predict(words.transform([message]))[0])

**What you will see:** SPAM. Now type a normal message like *Can you share the notes?*. Nobody wrote a rule like *if the message says free, it is spam*: the computer learned it from 12 examples.

## Your turn: Prediction or Generation?
For each task, decide P (prediction) or G (generation). Then open the answer.
1. Is this email spam?
2. Write a birthday poem
3. Tomorrow's temperature
4. Draw a logo
5. Which video should play next?

In [ ]:
#@title Show answer { display-mode: "form" }
print('1 P, 2 G, 3 P, 4 G, 5 P (a recommendation is a prediction of what you will like)')

### Quick check
1. Is face unlock prediction or generation?
2. Why should you check AI output?

In [ ]:
#@title Show answers { display-mode: "form" }
print("1. Prediction: it decides 'is this your face?'")
print('2. AI can be confidently wrong.')

# Stage 2: The AI Map
> **In one line:** AI is the big family; Machine Learning learns from examples; Deep Learning uses many layers; LLMs are deep learning for language.

### Easy notes
Think of it like circles inside circles:

**AI** > **Machine Learning** > **Deep Learning** > **LLMs** (like ChatGPT)

| Term | Easy meaning | Example |
|---|---|---|
| AI | computers doing smart tasks | all of the below |
| Machine Learning (ML) | learns patterns from examples | spam catcher |
| Deep Learning | ML with many layers, needs lots of data | face unlock |
| LLM | deep learning for language | ChatGPT |
| Generative AI | creates new content | image generators |
| Small language model | smaller, faster, cheaper LLM | AI on a phone |
| MLOps | keeping an ML model healthy after launch | like servicing a bike |
| AIOps | AI watching computer systems | like a guard watching CCTV |

**Rules vs Learning**
- **Rule:** a person writes it: `if temperature > 38: fever`
- **Learning:** we give examples, the computer finds the rule itself.

**Three ways a machine learns**
- **Supervised:** learns from examples with answers (spam / not spam)
- **Unsupervised:** finds groups on its own (types of customers)
- **Reinforcement:** learns by reward and punishment (a game-playing AI)

## Try it 1: Will this student pass?
The model learns from 10 past students. Move the sliders and run.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text
past_students = [[1, 55], [2, 62], [1.5, 72], [2.5, 48], [0.5, 65], [4, 80], [5, 92], [3.5, 85], [4.5, 70], [3, 90]]
result =        ["Fail", "Fail", "Fail",    "Fail",    "Fail",    "Pass",  "Pass",  "Pass",    "Pass",    "Pass"]
model = DecisionTreeClassifier(max_depth=2, random_state=0).fit(past_students, result)

study_hours = 1   #@param {type:"slider", min:0, max:8, step:0.5}
attendance = 50   #@param {type:"slider", min:30, max:100, step:5}
print("Prediction:", model.predict([[study_hours, attendance]])[0])
print("\nThe rule the model learned by itself:")
print(export_text(model, feature_names=["study_hours", "attendance"]))

**What you will see:** Fail for (1 hour, 50%). Try (5 hours, 90%). Look at the rule: the computer found it from examples. Nobody typed it.

## Try it 2: AIOps and MLOps in 10 lines

In [ ]:
cpu = [22, 25, 24, 27, 26, 93, 28, 25]          # server CPU % every 10 minutes
normal = sum(cpu[:5]) / 5
print("AIOps (watching a server):")
for minute, value in enumerate(cpu):
    print(f"  {minute*10:3d} min  {value:3d}%  {'ALERT!' if value > normal * 2 else 'ok'}")

accuracy = [0.92, 0.91, 0.88, 0.83, 0.76]       # a spam model's accuracy each week
print("\nMLOps (watching a model):")
for week, acc in enumerate(accuracy, 1):
    print(f"  week {week}  {acc:.0%}  {'RETRAIN' if acc < 0.80 else 'ok'}")

**What you will see:** one ALERT at minute 50 (the server went crazy) and RETRAIN in week 5 (the model got worse because spammers changed their tricks).

## Your turn: Where does it fit?
Face unlock · YouTube recommendations · ChatGPT writing code · Abnormal server behaviour · Monitoring a deployed model

In [ ]:
#@title Show answer { display-mode: "form" }
print('Face unlock: Deep learning (vision)')
print('YouTube: Machine learning')
print('ChatGPT code: LLM / Generative AI')
print('Server: AIOps')
print('Monitoring a model: MLOps')

### Quick check
1. Is every LLM a deep-learning model?
2. Which learning type finds groups without answers?

In [ ]:
#@title Show answers { display-mode: "form" }
print('1. Yes')
print('2. Unsupervised learning')

# Stage 3: Free Models
> **In one line:** a model is the **engine** inside an AI app, and you can run a free one yourself.

### Easy notes
**Model = engine.** The app is the car; the model is the engine. Your question goes in, an answer comes out.

**Why use a free model?**
- learn how AI really works
- no payment needed for practice
- your data can stay on your computer (if you run it locally)

**But free is not automatically better.** A small free model is fast and cheap, but may make more mistakes.

**What does a model need to run?** The model + a processor (CPU/GPU) + memory (RAM) + software. Today's model, **Qwen2.5-0.5B-Instruct**, is small (about half a billion parameters), so it runs on free Colab.

**Hosted vs Local**

| Hosted (like ChatGPT website) | Local (on your computer) |
|---|---|
| easy to start | needs setup |
| needs internet | can work offline |
| company controls it | you control it |

*Programme note: Laya is the free alternative named for this course. [Instructor: add details]*

## Try it 1: Chat with a free AI model
Type your question in the box and run. The first run downloads the model (about 1 GB), so wait a minute.

In [ ]:
question = "Explain machine learning in 3 simple points."  #@param {type:"string"}
reply, seconds = ask(question)
print(f"Answered in {seconds} seconds:\n\n{reply}")

**What you will see:** a short answer and the time it took. On a free CPU it may take several seconds: speed matters when you choose a model.

## Try it 2: Test the model like an engineer
Three quick tests. Read each answer and decide: correct or not?

In [ ]:
tests = {
    "Explain": "Explain neural networks to a 10-year-old in 2 lines.",
    "Maths":   "A student studies 2 hours a day for 5 days. How many hours in total? Give only the number.",
    "Rewrite": "Make this polite and professional: I can't come to class tomorrow.",
}
for name, q in tests.items():
    reply, seconds = ask(q, words=80)
    print(f"--- {name} ({seconds}s)\n{reply}\n")

**What you will see:** the Maths answer should be 10. Small models sometimes get simple maths wrong. Is a free model good enough for every job? No, it depends on the job.

## Your turn
Which matters most for each? (quality, speed or low cost)
1. An offline notes app on a phone
2. Checking exam answers

In [ ]:
#@title Show answer { display-mode: "form" }
print('1. Speed and low resources (it must run on a phone)')
print('2. Quality (mistakes are costly)')

### Explore more: run a model on your own laptop (optional)
1. Download Ollama from https://ollama.com/download (Linux: `curl -fsSL https://ollama.com/install.sh | sh`).
2. Check it works: `ollama -v`
3. Pick a small model at https://ollama.com/library and run `ollama run <model-name>`.

### Quick check
1. What 4 things does a model need to run?
2. Is the biggest model always the best?

In [ ]:
#@title Show answers { display-mode: "form" }
print('1. The model, a processor, memory and software')
print('2. No. Choose by the job, speed and cost.')

# Stage 4: How LLMs Work
> **In one line:** an LLM breaks your text into **tokens**, turns them into **numbers**, looks at the **context**, and **predicts the next word, again and again**.

### Easy notes (follow one prompt: *"Explain machine learning using cricket."*)
| Step | Easy meaning | Everyday comparison |
|---|---|---|
| 1. Tokens | text is cut into small pieces | like cutting a sentence into Lego blocks |
| 2. Numbers | each piece becomes a list of numbers | computers only understand numbers |
| 3. Context | everything the model can see right now | the question plus the chat so far |
| 4. Attention | which words matter most for each word | "cricket" tells the model *how* to explain |
| 5. Prediction | guess the next word, add it, repeat | like auto-complete on your phone, many times |

**Context window:** how much text the model can see at once. Think of a whiteboard: when it is full, the oldest writing gets erased.

**Temperature:** low = safe, predictable answers; high = more creative, more random.

**Training:** the model read huge amounts of text and kept correcting itself. Practice, feedback, improve, like learning cricket shots.

**Fine-tuning:** taking a trained model and training it a little more for one job. Like an MBBS doctor becoming a cardiologist. Example: the happy/sad model in Stage 1 is a general model fine-tuned on movie reviews.

## Try it 1: See the tokens

In [ ]:
ask("hi", words=5)                      # makes sure the model is loaded
tok, brain = _chat.tokenizer, _chat.model

text = "Explain machine learning using cricket"  #@param {type:"string"}
ids = tok(text)["input_ids"]
pieces = [tok.decode([i]) for i in ids]
print("Tokens :", pieces)
print("Numbers:", ids)
print(f"{len(text.split())} words became {len(ids)} tokens")
print("This model can see up to", brain.config.max_position_embeddings, "tokens at once (its context window)")

**What you will see:** each token and its ID number. Now type your own name or a sentence in Telugu or Hindi: you usually get more tokens than words.

## Try it 2: Watch the model guess the next word

In [ ]:
import torch
start = "Machine learning is"  #@param {type:"string"}
ids = tok(start, return_tensors="pt")
with torch.no_grad():
    scores = brain(**ids).logits[0, -1]
chances = torch.softmax(scores, dim=-1)
top = torch.topk(chances, 5)
print(f"After '{start}', the model's top 5 guesses:")
for p, i in zip(top.values, top.indices):
    print(f"  {tok.decode([int(i)])!r:15} {p.item():.0%}")

**What you will see:** five possible next words with their chances. Writing an answer is just this step repeated: pick a word, add it, guess again.

## Try it 3: Same model, three prompts

In [ ]:
for prompt in ["Explain machine learning.",
               "Explain machine learning using cricket.",
               "Explain machine learning using cricket in exactly 3 lines."]:
    reply, seconds = ask(prompt, words=120)
    print(f"=== {prompt}\n{reply}\n")

**What you will see:** three different answers from the same model. What changed? Only the prompt (the context). Better prompt, better answer.

### Explore more (no code, in your browser)
- **Tokenizer Playground** https://huggingface.co/spaces/Xenova/the-tokenizer-playground : paste a sentence and count its tokens.
- **Transformer Explainer** https://poloclub.github.io/transformer-explainer/ : type *Machine learning is*, see the next-word chances, then move the temperature slider.
- **TensorFlow Playground** http://playground.tensorflow.org : press Play and watch a small network learn (training).

### Quick check
1. Why does text become numbers?
2. What happens when the context window is full?

In [ ]:
#@title Show answers { display-mode: "form" }
print('1. Computers can only calculate with numbers.')
print('2. The oldest text drops out, like a full whiteboard being erased.')

# Stage 5: Benchmarks
> **In one line:** a benchmark is a **common exam paper** for AI models, so we compare them with scores instead of opinions.

### Easy notes
**Benchmark:** same questions, same marking, compare the scores.

| Score | Easy meaning | Best for |
|---|---|---|
| **Accuracy** | correct answers / total answers | yes-or-no tasks (spam or not) |
| **pass@k** | did at least 1 of k tries work? | coding (run it, keep the one that works) |
| **Human votes** | people pick the better answer | chat and explanations |

**How to read a model comparison table**
1. Which test? 2. Which score? 3. Compare on the same test only. 4. Look for trade-offs. 5. No model wins everything.

**Remember:** the best model depends on **your** task.

## Try it 1: Accuracy calculator

In [ ]:
correct = 18  #@param {type:"integer"}
total = 25    #@param {type:"integer"}
print(f"Accuracy = {correct}/{total} = {correct/total:.0%}")

**What you will see:** 72%. Try the slide example: 7 out of 10.

## Try it 2: pass@k

In [ ]:
tries = ["wrong", "wrong", "correct", "wrong", "wrong"]   # 5 attempts at a coding problem
for k in [1, 2, 3, 5]:
    passed = "correct" in tries[:k]
    print(f"pass@{k}: {'PASS' if passed else 'FAIL'}")

**What you will see:** FAIL for k = 1 and 2, PASS from k = 3. More tries means more chances, so pass@k always says which k.

## Try it 3: Choose a model for a job
The numbers are made up for practice. Pick a job and run.

In [ ]:
table = pd.DataFrame({"Accuracy": [82, 86, 84], "pass@5": [61, 74, 81], "Human votes": [48, 55, 63]},
                     index=["Alpha", "Beta", "Gamma"])
print(table, "\n")
job = "coding assistant"  #@param ["spam filter", "coding assistant", "student tutor"]
important = {"spam filter": "Accuracy", "coding assistant": "pass@5", "student tutor": "Human votes"}[job]
print(f"For a {job}, the key score is {important} -> choose {table[important].idxmax()}")
table.plot(kind="bar", rot=0, title="Made-up benchmark scores"); plt.show()

**What you will see:** Beta wins accuracy, Gamma wins pass@5 and human votes. No model wins everything, so the job decides.

## Your turn
A model gets 30 of 40 right. Accuracy? A coding model works on try 4 of 5. Does pass@3 pass?

In [ ]:
#@title Show answer { display-mode: "form" }
print('Accuracy = 30/40 = 75%')
print('pass@3: FAIL (the first 3 tries were wrong); pass@5 would PASS')

### Explore more
**LMArena** https://lmarena.ai : ask one question, read two hidden answers, vote for the better one. That is how human votes are collected.

### Quick check
1. Which score fits a coding assistant?
2. Does one high score mean a model is best at everything?

In [ ]:
#@title Show answers { display-mode: "form" }
print('1. pass@k')
print('2. No. It only tells you about that one test.')

# Stage 6: GitHub Skills (AI agent skills)
> **In one line:** a skill is a **recipe card** for an AI agent: the same model follows your steps every time.

### Easy notes
**Skill = a folder with a `SKILL.md` file.** It has:
- `name`: short name, same as the folder (for example `code-review`)
- `description`: what it does and **when** to use it (the agent reads this to decide)
- the instructions: your step-by-step checklist

**How an agent uses it:** task arrives > agent finds a skill whose description matches > reads the instructions > does the job that way.

**Where it lives:** in your GitHub repo under `.github/skills/<name>/SKILL.md`. GitHub Copilot (cloud agent, CLI, VS Code agent mode) can use it.

**With a free model:** a plain chat does not load skills automatically, so we paste the skill text into the prompt ourselves. Same idea.

**Not the same thing:** *GitHub Skills* (skills.github.com) is a free course website that teaches people how to use GitHub.

**Safety:** never put real passwords or keys in code or prompts. Read any skill made by someone else before installing it.

## Try it 1: Write your first skill

In [ ]:
skill = """---
name: code-review
description: Reviews beginner Python code. Use when asked to review code.
---
# Code Review Skill
1. Check if the code works correctly.
2. Find possible bugs.
3. Suggest one improvement.
4. Say if tests are needed.
Answer as a numbered list.
"""
os.makedirs(".github/skills/code-review", exist_ok=True)
open(".github/skills/code-review/SKILL.md", "w").write(skill)
print("Saved .github/skills/code-review/SKILL.md")
print("Has a name?        ", "name:" in skill)
print("Has a description? ", "description:" in skill)

**What you will see:** the file is saved with a name and a description. Both are required.

## Try it 2: Same model, with and without the skill

In [ ]:
my_code = """
API_KEY = "demo-secret-123"
x = input("Number: ")
print(int(x) * 2)
"""
plain, _ = ask("Review this Python code:" + my_code, words=150)
steps = skill.split("---")[-1]                 # the instructions part of the skill
with_skill, _ = ask("Review this Python code:" + my_code, rules=steps, words=200)
print("=== WITHOUT skill ===\n" + plain + "\n")
print("=== WITH skill ===\n" + with_skill)

**What you will see:** the answer with the skill follows your numbered checklist. A small model will not be perfect, but the shape of the answer changes. (The key is fake, for class only.)

## Your turn: design a skill
Pick one idea: testing, documentation, security. Write a name, a description that says *when* to use it, and 3 to 5 steps. Then copy Try it 1 and Try it 2 to test it.

In [ ]:
#@title Show an example { display-mode: "form" }
print('name: security-check')
print('description: Finds basic security mistakes in Python. Use for security reviews.')
print('1. Look for passwords or keys in the code.')
print('2. Check user input is validated.')
print('3. Report: problem, risk, fix.')

### Explore more
- Save your skill on GitHub: create a repo, **Add file > Create new file**, type `.github/skills/code-review/SKILL.md`, paste, **Commit**.
- Ready-made skills (read before using!): https://awesome-copilot.github.com/skills/
- GitHub's guide: https://docs.github.com/en/copilot/how-tos/copilot-on-github/customize-copilot/customize-cloud-agent/add-skills
- Learn GitHub itself: https://skills.github.com

### Quick check
1. Which two fields must a SKILL.md have?
2. How does an agent decide to use a skill?

In [ ]:
#@title Show answers { display-mode: "form" }
print('1. name and description')
print("2. It matches your task with the skill's description.")

# Stage 7: RAG + OKF
> **In one line:** RAG is an **open-book exam** for AI: first find the right page in your notes, then answer from it.

### Easy notes
**The problem:** the model has never seen your college's documents, so it cannot answer *"What is our attendance policy?"*.

**RAG = Retrieve, then Generate**
1. **Retrieve:** find the part of your notes that matches the question
2. **Generate:** give that part to the model and let it answer

**Embeddings:** each piece of text becomes a list of numbers that captures its *meaning*, like GPS coordinates for meaning. *"How many leave days?"* and *"What is the leave limit?"* end up close together, even with different words.

**Keyword search vs meaning search**
- Keyword search matches **exact words**, so it misses different wording.
- Meaning search (embeddings) matches **similar ideas**, but is not always right, so always check what it found.

**OKF (Open Knowledge Format)** from Google Cloud: keep knowledge as a folder of Markdown notes, each starting with a short header. The only required field is `type`. One topic per file and clear `##` headings make notes easy to search.

All notes below are **made up** for practice.

## Try it 1: Write your notes (OKF style)

In [ ]:
NOTES = {
"knowledge/academics/attendance.md": """---
type: Policy
title: Attendance
---
# Attendance
## Requirement
Students should maintain the required attendance according to institutional rules.
## Exemptions
Attendance exemptions may be considered under specific circumstances.""",
"knowledge/academics/exams.md": """---
type: Policy
title: Examinations
---
# Examinations
## Schedule
Examination dates are announced on the official notice board.
## Rules
Students must carry their ID card to the examination hall.""",
"knowledge/placements/eligibility.md": """---
type: Guide
title: Placement Eligibility
---
# Placement Eligibility
## Criteria
Students should meet the minimum academic criteria announced by the placement cell.
## Documents
Keep your resume and ID proof ready.""",
}
for path, text in NOTES.items():
    os.makedirs(os.path.dirname(path), exist_ok=True)
    open(path, "w").write(text)
    print("saved", path, "| has type:", "type:" in text)

**What you will see:** three notes saved, each with a `type`. Open the folder icon on the left of Colab to see them.

## Try it 2: Ask questions from your notes

In [ ]:
from sentence_transformers import SentenceTransformer
meaning = SentenceTransformer("all-MiniLM-L6-v2")      # turns text into meaning-numbers

def load_pieces():
    pieces = []
    for path in sorted(glob.glob("knowledge/**/*.md", recursive=True)):
        text = open(path).read()
        title = re.search(r"^# (.+)$", text, re.M).group(1)
        for part in text.split("## ")[1:]:             # one piece per ## section
            pieces.append({"file": path, "text": f"{title}: {part.strip()}", "status": "draft" if "status: draft" in text else ""})
    return pieces

PIECES = load_pieces()
VECTORS = meaning.encode([p["text"] for p in PIECES], normalize_embeddings=True)

def ask_notes(question):
    q = meaning.encode([question], normalize_embeddings=True)[0]
    best = PIECES[int(np.argmax(VECTORS @ q))]
    print("FOUND IN:", best["file"], "\n  ", best["text"])
    reply, _ = ask(f"Notes: {best['text']}\nQuestion: {question}",
                   rules="Answer only from the notes. If the notes do not say, reply: I don't know.", words=80)
    print("ANSWER:", reply)

question = "What does the attendance document say?"  #@param {type:"string"}
ask_notes(question)

**What you will see:** first **what it found** (so you can check it), then the answer. Try *When are exams announced?*, *What documents do I need for placements?* and *What is the hostel fee?* (not in your notes, so it should say it doesn't know).

## Try it 3: Keyword search vs meaning search

In [ ]:
question = "Can I miss classes because of a medical reason?"
q_words = set(question.lower().replace("?", "").split())
print("KEYWORD search (shared words):")
for p in PIECES:
    shared = q_words & set(p["text"].lower().split())
    print(f"  {len(shared)} shared words -> {p['file']}")
q = meaning.encode([question], normalize_embeddings=True)[0]
print("\nMEANING search picks:", PIECES[int(np.argmax(VECTORS @ q))]["file"])

**What you will see:** almost no shared words anywhere, so keyword search is stuck. Meaning search should pick the attendance note, because *miss classes* and *attendance exemptions* mean similar things.

### Explore more
- OKF specification: https://github.com/GoogleCloudPlatform/knowledge-catalog/blob/main/okf/SPEC.md
- Save your notes on GitHub: **Add file > Create new file**, type `knowledge/academics/attendance.md`, paste, **Commit**.

### Quick check
1. RAG in one line?
2. Why print what was found before the answer?

In [ ]:
#@title Show answers { display-mode: "form" }
print('1. Find the right part of your notes first, then answer from it.')
print('2. So you can check the answer is based on the right note.')

# Stage 8: Night Project, Knowledge RAG Agent
> **Goal:** an agent that answers from your notes, warns when a note is not trustworthy, and gets a **measured** score.

### The plan (commit to GitHub after each step)
1. **Notes:** at least 5 notes in `knowledge/`
2. **V1:** keyword search (works with no AI model)
3. **V2:** meaning search (embeddings)
4. **Trust check:** if a note says `status: draft`, warn instead of answering confidently *(our own simple rule)*
5. **Test:** 10 questions, a score, and one failure explained
6. **Demo:** show it live and tell its story

**Rubric (100):** works live 25 · you understand it 25 · measured, not claimed 25 · honest about limits 15 · clear story 10.
*[Instructor: confirm the hand-in format.]*

## Step 1: Add two more notes (one is a draft)

In [ ]:
MORE = {
"knowledge/campus/library.md": "---\ntype: Policy\ntitle: Library\n---\n# Library\n## Timings\nThe library is open on working days during college hours.\n## Borrowing\nStudents may borrow books using their ID card.",
"knowledge/campus/hostel.md": "---\ntype: Policy\ntitle: Hostel\nstatus: draft\n---\n# Hostel\n## Rooms\nHostel rooms are allotted by the hostel office.\n## Visitors\nVisitors are allowed only during visiting hours.",
}
for path, text in MORE.items():
    os.makedirs(os.path.dirname(path), exist_ok=True); open(path, "w").write(text); print("saved", path)

## Step 2: The agent (V1 keyword search + trust check)
This version needs **no AI model**, so it always works, even offline.

In [ ]:
SKIP = {"i", "a", "an", "the", "of", "to", "is", "are", "can", "my", "do", "what", "when", "how", "for", "in", "on", "and", "should", "where"}

def load_pieces():
    pieces = []
    for path in sorted(glob.glob("knowledge/**/*.md", recursive=True)):
        text = open(path).read()
        title = re.search(r"^# (.+)$", text, re.M).group(1)
        for part in text.split("## ")[1:]:
            pieces.append({"file": path, "text": f"{title}: {part.strip()}", "status": "draft" if "status: draft" in text else ""})
    return pieces

def word_set(t):
    return set(re.findall(r"[a-z]+", t.lower())) - SKIP

def agent(question, show=True):
    q = word_set(question)
    score, best = max(((len(q & word_set(p["text"])) / max(len(q), 1), p) for p in PIECES), key=lambda x: x[0])
    if score < 0.2:
        reply, file = "I could not find this in my notes.", None
    elif best["status"] == "draft":
        reply, file = "WARNING: this note is a draft, please confirm with the office. " + best["text"], best["file"]
    else:
        reply, file = best["text"], best["file"]
    if show: print(f"Q: {question}\nA: {reply}\n")
    return file

PIECES = load_pieces()
print(len(PIECES), "pieces loaded\n")
agent("When are exam dates announced?")
agent("When can visitors come to the hostel?")

**What you will see:** a normal answer for exams, and a WARNING for the hostel because that note is a draft.

## Step 3: Measure it on 10 questions

In [ ]:
TESTS = [("What is the attendance requirement?", "knowledge/academics/attendance.md"),
         ("Can attendance exemptions be given?", "knowledge/academics/attendance.md"),
         ("When are exam dates announced?", "knowledge/academics/exams.md"),
         ("What must I carry to the exam hall?", "knowledge/academics/exams.md"),
         ("What documents do I need for placements?", "knowledge/placements/eligibility.md"),
         ("Who announces the placement criteria?", "knowledge/placements/eligibility.md"),
         ("When is the library open?", "knowledge/campus/library.md"),
         ("How do I borrow a book?", "knowledge/campus/library.md"),
         ("Can I miss classes for a medical reason?", "knowledge/academics/attendance.md"),
         ("What is the canteen menu today?", None)]
score = 0
for q, expected in TESTS:
    got = agent(q, show=False)
    ok = got == expected
    score += ok
    print(("PASS " if ok else "FAIL ") + q + ("" if ok else f"   (expected {expected}, got {got})"))
print(f"\nSCORE: {score}/{len(TESTS)}")

**What you will see:** a real score. The *medical reason* question should fail with keyword search, because it shares no words with the attendance note. That is your failure to explain, and V2 (meaning search from Stage 7) is your fix.

## Step 4: Upgrade to V2 (meaning search)
Replace the matching line in `agent()` with the meaning search from **Stage 7, Try it 2**, run the 10 tests again and compare the scores.

**Write down for your demo:** your V1 score, your V2 score, one question that failed, why it failed, and how you fixed it.

# All links in one place
| Link | Use it for |
|---|---|
| https://colab.research.google.com | running this lab |
| https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct | the free model we used |
| https://ollama.com/download | running a model on your laptop |
| https://huggingface.co/spaces/Xenova/the-tokenizer-playground | seeing tokens |
| https://poloclub.github.io/transformer-explainer/ | seeing attention and next-word guesses |
| http://playground.tensorflow.org | watching a network learn |
| https://lmarena.ai | voting between two AI answers |
| https://skills.github.com | learning GitHub |
| https://awesome-copilot.github.com/skills/ | ready-made agent skills |
| https://github.com/GoogleCloudPlatform/knowledge-catalog/blob/main/okf/SPEC.md | the OKF format |

**Safety:** made-up data only, fake secrets only, read any skill or script before running it.